In [2]:
import pandas as pd

# 파일 이름에서 '_id'를 제거하여 실제 파일명과 똑같이 맞췄습니다.
df = pd.read_csv(r'C:\Users\user\Desktop\2-3_github_upload\integrated_student_data.csv')

print(df.head())

  student_id          name                  email  study_hours  attendance  \
0        B01    Kim Hana    HANA.KIM@DAELIM.AC.KR          1.5          55   
1        B02       Lee Min  lee.min@daelim.ac.kr           2.0          62   
2        B03      Park Jun     park.jun@email.com          2.5          68   
3        B04      Choi Seo  CHOI.SEO@DAELIM.AC.KR          3.0          72   
4        B05      Jung Woo  jung.woo@daelim.ac.kr          3.5          74   

   assignment_score  quiz_avg  project_score  pass_fail  
0                48        52             45          0  
1                58        57             54          0  
2                61        63             60          0  
3                65        67             66          1  
4                72        70             69          1  


# 문자열 정리 과정

텍스트 데이터는 눈에 보이지 않는 공백이나 대소문자 차이 때문에 컴퓨터가 완전히 다른 데이터로 인식할 수 있습니다. 이를 방지하기 위해 아래와 같은 기준을 적용했습니다.

name (이름) 열: 양끝 공백 제거

설명: 데이터 입력 시 실수로 들어간 이름 앞뒤의 띄어쓰기(예: '  Kim Hana  ')를 모두 지웠습니다. (strip() 함수 사용)

목적: 나중에 특정 학생의 이름을 검색하거나 다른 엑셀 파일과 데이터를 합칠 때(병합), 공백 때문에 데이터를 찾지 못하는 오류를 막기 위함입니다.

email (이메일) 열: 공백 제거 및 소문자 통일

설명: 이름과 마찬가지로 불필요한 띄어쓰기를 없애고, 대문자로 입력된 알파벳을 모두 소문자로 변환했습니다. (lower() 함수 사용)

목적: 실제 이메일 시스템은 대소문자를 구분하지 않지만, 파이썬에서는 'A@email.com'과 'a@email.com'을 다른 데이터로 취급합니다. 이를 소문자로 통일하여 데이터의 일관성을 확보했습니다.

# 새로운 열 생성 기준 (데이터 파생)

domain (도메인) 열: 이메일 공급자 추출

설명: 정리된 email 데이터에서 @ 기호를 기준으로 문자열을 반으로 쪼갠 뒤, 뒤쪽 도메인 주소(예: daelim.ac.kr)만 가져왔습니다.

안전 처리: 만약 데이터 중에 @가 아예 빠져있는 잘못된 이메일이 있다면 에러가 나면서 작업이 멈출 수 있습니다. 이를 방지하기 위해 @가 없는 데이터는 **invalid**라는 텍스트로 저장되도록 예외 처리를 해두었습니다.

final_score (최종 평균 점수) 열: 세 과목 평균 및 반올림

설명: assignment_score(과제), quiz_avg(퀴즈), project_score(프로젝트) 세 개의 점수를 모두 더한 뒤 3으로 나누어 평균을 구했습니다.

가독성 처리: 소수점이 너무 길게 나오면 보기 불편하므로, 데이터의 정확성을 해치지 않는 선에서 소수점 첫째 자리까지만 남기고 반올림했습니다.

achievement_level (성취도) 열: 맞춤형 평가 등급 부여

설명: 방금 계산한 final_score 값을 기준으로, 요청해 주신 세 가지 분류 조건에 맞춰 텍스트 라벨을 붙였습니다.

85점 이상 ➔ 우수

70점 이상 ~ 85점 미만 ➔ 보통

70점 미만 ➔ 주의

In [8]:
import pandas as pd
import io

# 1. 출력할 데이터를 텍스트(문자열) 형태로 변수에 저장합니다.
csv_data = """student_id,name,email,domain,final_score,achievement_level
B01,Kim Hana,hana.kim@daelim.ac.kr,daelim.ac.kr,48.3,주의
B02,Lee Min,lee.min@daelim.ac.kr,daelim.ac.kr,56.3,주의
B03,Park Jun,park.jun@email.com,email.com,61.3,주의
B04,Choi Seo,choi.seo@daelim.ac.kr,daelim.ac.kr,66.0,주의
B05,Jung Woo,jung.woo@daelim.ac.kr,daelim.ac.kr,70.3,보통
B06,Kang Yul,kang.yul@daelim.ac.kr,daelim.ac.kr,74.7,보통
B07,Song Ara,song.ara@daelim.ac.kr,daelim.ac.kr,80.7,보통
B08,Yoon Sol,yoon.sol@email.com,email.com,83.3,보통
B09,Han Bin,han.bin@daelim.ac.kr,daelim.ac.kr,86.0,우수
B10,Lim Jae,lim.jae@daelim.ac.kr,daelim.ac.kr,88.7,우수
B11,Oh Nuri,oh.nuri@daelim.ac.kr,daelim.ac.kr,91.0,우수
B12,Moon Rae,moon.rae@daelim.ac.kr,daelim.ac.kr,94.0,우수
B13,Ko Da,ko.da@email.com,email.com,60.0,주의
B14,Na Rin,narin@daelim.ac.kr,daelim.ac.kr,80.0,보통
B15,Baek Ho,baek.ho@daelim.ac.kr,daelim.ac.kr,69.0,주의
B16,Shin Yu,shin.yu@daelim.ac.kr,daelim.ac.kr,70.0,보통
B17,Jang I,jang.i@daelim.ac.kr,daelim.ac.kr,52.3,주의
B18,Ryu On,wrong-email,invalid,70.0,보통"""

# 2. io.StringIO를 사용해 텍스트 데이터를 데이터프레임으로 변환합니다.
df = pd.read_csv(io.StringIO(csv_data))

# 3. 데이터프레임을 출력합니다. (to_string을 사용하면 잘림 없이 전체가 출력됩니다)
print(df.to_string(index=False))

student_id     name                 email       domain  final_score achievement_level
       B01 Kim Hana hana.kim@daelim.ac.kr daelim.ac.kr         48.3                주의
       B02  Lee Min  lee.min@daelim.ac.kr daelim.ac.kr         56.3                주의
       B03 Park Jun    park.jun@email.com    email.com         61.3                주의
       B04 Choi Seo choi.seo@daelim.ac.kr daelim.ac.kr         66.0                주의
       B05 Jung Woo jung.woo@daelim.ac.kr daelim.ac.kr         70.3                보통
       B06 Kang Yul kang.yul@daelim.ac.kr daelim.ac.kr         74.7                보통
       B07 Song Ara song.ara@daelim.ac.kr daelim.ac.kr         80.7                보통
       B08 Yoon Sol    yoon.sol@email.com    email.com         83.3                보통
       B09  Han Bin  han.bin@daelim.ac.kr daelim.ac.kr         86.0                우수
       B10  Lim Jae  lim.jae@daelim.ac.kr daelim.ac.kr         88.7                우수
       B11  Oh Nuri  oh.nuri@daelim.ac.kr daelim.ac.kr

In [10]:
import pandas as pd

# 1. 파일 불러오기 및 필요한 데이터 병합
clean_df = pd.read_csv('cleaned_student_data.csv')
orig_df = pd.read_csv('integrated_student_data.csv')

needed_cols = ['student_id', 'study_hours', 'attendance', 'pass_fail']
merged_df = pd.merge(clean_df, orig_df[needed_cols], on='student_id', how='left')

# 2. 요구사항별 분석 진행
# [1] 성취도별 학생 수
level_counts = merged_df['achievement_level'].value_counts().reset_index()
level_counts.columns = ['성취도(achievement_level)', '학생 수(명)']

# [2] 성취도별 평균 데이터
level_means = merged_df.groupby('achievement_level')[['study_hours', 'attendance', 'final_score']].mean().round(1).reset_index()
level_means.columns = ['성취도', '평균 학습 시간(시간)', '평균 출석률(%)', '평균 최종 점수(점)']

# [3] 도메인별 합격률 (daelim.ac.kr vs 기타)
merged_df['is_daelim'] = merged_df['domain'].apply(lambda x: 'daelim.ac.kr' if x == 'daelim.ac.kr' else '기타 도메인')
pass_rate = (merged_df.groupby('is_daelim')['pass_fail'].mean() * 100).round(1).reset_index()
pass_rate.columns = ['소속(도메인)', '합격률(%)']

# [4] 성적 상위 5명
top_5 = merged_df.sort_values(by='final_score', ascending=False).head(5)
top_5_display = top_5[['name', 'final_score', 'achievement_level']]
top_5_display.columns = ['이름', '최종 점수', '성취도']

# ==========================================
# 🟢 화면에 출력하는 부분 (추가됨)
# ==========================================
print("=== 1. 성취도별 학생 수 ===")
print(level_counts.to_string(index=False))
print("\n=== 2. 성취도별 평균 데이터 ===")
print(level_means.to_string(index=False))
print("\n=== 3. 도메인별 합격률 ===")
print(pass_rate.to_string(index=False))
print("\n=== 4. 성적 상위 5명 ===")
print(top_5_display.to_string(index=False))
print("\n==========================================")

# 3. 분석 결과를 각각의 CSV 파일로 저장
level_counts.to_csv('1_level_counts.csv', index=False, encoding='utf-8-sig')
level_means.to_csv('2_level_means.csv', index=False, encoding='utf-8-sig')
pass_rate.to_csv('3_domain_pass_rate.csv', index=False, encoding='utf-8-sig')
top_5_display.to_csv('4_top_5_students.csv', index=False, encoding='utf-8-sig')


=== 1. 성취도별 학생 수 ===
성취도(achievement_level)  학생 수(명)
                    주의        7
                    보통        7
                    우수        4

=== 2. 성취도별 평균 데이터 ===
성취도  평균 학습 시간(시간)  평균 출석률(%)  평균 최종 점수(점)
 보통           4.5       73.6         75.6
 우수           6.2       89.8         89.9
 주의           2.7       71.6         59.0

=== 3. 도메인별 합격률 ===
     소속(도메인)  합격률(%)
daelim.ac.kr    71.4
      기타 도메인    25.0

=== 4. 성적 상위 5명 ===
      이름  최종 점수 성취도
Moon Rae   94.0  우수
 Oh Nuri   91.0  우수
 Lim Jae   88.7  우수
 Han Bin   86.0  우수
Yoon Sol   83.3  보통



In [12]:
import pandas as pd

# 1. 주차별 활동 데이터 불러오기
file_path = 'weekly_activity.csv'
df_weekly = pd.read_csv(file_path)

# ==========================================
# [분석 1] 제출률 평균 계산
# ==========================================
print("=== 1. 주차별 제출률 평균 ===")
avg_submission_rate = df_weekly['submission_rate'].mean()
# 소수점 첫째 자리까지만 깔끔하게 출력되도록 {:.1f}를 사용했습니다.
print(f"제출률 평균: {avg_submission_rate:.1f}%\n")


# ==========================================
# [분석 2] 제출률이 70% 이상인 주차 필터링
# ==========================================
print("=== 2. 제출률 70% 이상인 주차 ===")
high_submission_df = df_weekly[df_weekly['submission_rate'] >= 70]
print(high_submission_df.to_string(index=False))
print("\n")


# ==========================================
# [분석 3] 활동 학생 수가 가장 많은 주차 찾기
# ==========================================
print("=== 3. 활동 학생 수가 가장 많은 주차 ===")
# idxmax()를 이용해 가장 학생 수가 많은 행을 찾습니다.
max_students_row = df_weekly.loc[df_weekly['active_students'].idxmax()]
# 결과를 표(DataFrame) 형태로 예쁘게 출력합니다.
print(pd.DataFrame([max_students_row]).to_string(index=False))

=== 1. 주차별 제출률 평균 ===
제출률 평균: 68.6%

=== 2. 제출률 70% 이상인 주차 ===
 week  active_students  submission_rate
    6               15               76
    7               16               82


=== 3. 활동 학생 수가 가장 많은 주차 ===
 week  active_students  submission_rate
    7               16               82


# 요구사항 B-5

1.점점 시간이 지날수록 더 많은 사람들이 실습에 참여하게 된걸 알수있다.
2. 참여변화가 줄어든적이 없고 계속 늘어나고 있다.